# ISLES'24 core extraction, audit, and NCCT baseline staging

Run **Runtime → Run all**.

This notebook starts from the already-downloaded `train.7z` in Google Drive and prepares the first modern research baseline without extracting the full 99 GB dataset.

It selectively extracts the **core modeling subset** only: acute NCCT, registered CTA, registered Tmax/CBF/CBV/MTT maps, final-infarct labels, and baseline clinical CSVs. It deliberately excludes raw 4-D CTP, follow-up DWI/ADC, and post-treatment/outcome files from this stage.

The extraction is resumable in 10-subject batches. After extraction, the notebook audits all 149 cases and stages a persistent **NCCT-only nnU-Net v2 dataset** with deterministic five-fold patient splits.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, shutil, subprocess
from pathlib import Path

ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
CORE_DIR = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/core')
RESULTS_DIR = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/results')
NNUNET_RAW = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/nnUNet_raw')
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO_PARENT = Path('/content')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')

if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Expected completed ISLES24 archive: {ARCHIVE}')
print(f'Archive ready: {ARCHIVE.stat().st_size / (1024**3):.2f} GiB')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NNUNET_RAW.mkdir(parents=True, exist_ok=True)


## Install extraction tools and the latest repository snapshot


In [ ]:
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','p7zip-full','unzip'], check=True)

if REPO.exists():
    shutil.rmtree(REPO)
if REPO_ZIP.exists():
    REPO_ZIP.unlink()

subprocess.run([
    'curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o', str(REPO_ZIP)
], check=True)
subprocess.run(['unzip','-q',str(REPO_ZIP),'-d',str(REPO_PARENT)], check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-q','-e','.'], check=True)
print('Repository snapshot ready:', REPO)


## Selectively extract the core ISLES'24 modeling subset

Completed 10-subject extraction batches are skipped automatically if Colab disconnects.


In [ ]:
subprocess.run([
    'python','scripts/extract_isles2024_core.py',
    str(ARCHIVE), str(CORE_DIR),
    '--batch-size','10'
], check=True)


## Audit all 149 public training cases and NCCT-space geometry


In [ ]:
AUDIT_JSON = RESULTS_DIR / 'isles2024_core_audit.json'
subprocess.run([
    'python','scripts/audit_isles2024.py', str(CORE_DIR),
    '--channels','ncct','cta','tmax','cbf','cbv','mtt',
    '--json', str(AUDIT_JSON)
], check=True)
print('Audit written to:', AUDIT_JSON)


## Stage the first predeclared modern baseline: NCCT-only nnU-Net v2

The staging step is resumable. Only acute NCCT is copied as an input channel; the final-infarct mask is copied as the label. Follow-up MRI and outcome variables are never staged as inputs.


In [ ]:
subprocess.run([
    'python','scripts/stage_isles2024_nnunet.py',
    str(CORE_DIR), str(NNUNET_RAW),
    '--dataset-id','504',
    '--dataset-name','ISLES2024_NCCT',
    '--channels','ncct',
    '--mode','copy',
    '--resume',
    '--split-seed','2026'
], check=True)


## Verify persistent preparation outputs


In [ ]:
import json

DATASET_DIR = NNUNET_RAW / 'Dataset504_ISLES2024_NCCT'
audit = json.loads(AUDIT_JSON.read_text())
print('Audit case count:', audit.get('cases'))
print('Mask geometry matches NCCT:', audit.get('lesion_mask_matches_ncct'))
print('nnU-Net imagesTr:', len(list((DATASET_DIR / 'imagesTr').glob('*.nii.gz'))))
print('nnU-Net labelsTr:', len(list((DATASET_DIR / 'labelsTr').glob('*.nii.gz'))))
print('dataset.json exists:', (DATASET_DIR / 'dataset.json').exists())
print('splits_final.json exists:', (DATASET_DIR / 'splits_final.json').exists())
print('\nPersistent dataset:', DATASET_DIR)


## Reconnection rule

If Colab disconnects, reopen this notebook and choose **Runtime → Run all** again. Completed extraction batches and complete staged files are reused rather than restarted.
